In [1]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import OneHotEncoder
import os, psutil
import os.path as path
import gc
from pathlib import Path

In [2]:
print(os.getcwd())

D:\assetpringml


In [3]:
#stock-month lagged characteristics; url = https://dachxiu.chicagobooth.edu/download/datashare.zip
char = pd.read_csv('datashare.csv')
char.head(10)

,permno,DATE,mvel1,beta,betasq,chmom,dolvol,idiovol,indmom,mom1m,...,stdcf,ms,baspread,ill,maxret,retvol,std_dolvol,std_turn,zerotrade,sic2
0,10006,19570131,82249.000,1.122846,1.260784,0.047180,9.569953,0.025742,0.046433,0.044843,...,NaN,NaN,0.013234,9.411565e-08,0.015453,0.008058,0.355638,0.460420,1.120996e-07,37.0
1,10014,19570131,3903.375,0.426734,0.182102,-0.275641,6.237836,0.072103,0.046433,-0.086957,...,NaN,NaN,0.033305,6.610609e-06,0.047619,0.033495,1.152126,1.169610,9.229146e-08,NaN
2,10022,19570131,9273.250,1.066449,1.137313,-0.025490,7.008844,0.027648,0.046433,-0.060377,...,NaN,NaN,0.016023,2.286832e-06,0.020833,0.015589,0.815777,0.679803,1.181757e-07,NaN
3,10030,19570131,54465.875,0.926038,0.857547,0.018171,9.825337,0.021700,0.046433,0.044633,...,NaN,NaN,0.015295,1.464273e-07,0.039326,0.015849,0.739302,1.333656,6.126699e-08,NaN
4,10057,19570131,40250.000,1.247748,1.556875,0.025785,7.901007,0.025506,0.046433,0.086667,...,NaN,NaN,0.005954,1.380375e-06,0.056856,0.019945,0.755510,0.410391,3.315790e+00,NaN
5,10065,19570131,76945.250,1.264106,1.597964,0.193374,8.906529,0.023942,0.046433,0.077778,...,NaN,NaN,0.008934,1.393915e-07,0.022222,0.008179,0.561265,0.461107,1.687356e-07,NaN
6,10102,19570131,186193.500,1.445027,2.088104,-0.058910,10.375318,0.025526,0.046433,0.041237,...,NaN,NaN,0.009637,7.398559e-08,0.017632,0.008298,0.787710,0.773360,1.530290e-07,28.0
7,10137,19570131,225984.000,0.585546,0.342864,-0.090805,8.887653,0.018691,0.046433,0.018779,...,NaN,NaN,0.013150,1.679162e-07,0.023923,0.009137,0.597666,0.106998,6.616541e-07,49.0
8,10145,19570131,962994.375,1.132210,1.281900,-0.084218,10.850652,0.023319,0.046433,0.052846,...,NaN,NaN,0.016536,2.909367e-08,0.024324,0.013477,0.410964,0.174574,2.830609e-07,99.0
9,10153,19570131,279846.875,1.173314,1.376666,-0.008426,10.547950,0.021665,0.046433,0.117886,...,NaN,NaN,0.016482,3.379634e-08,0.027132,0.011101,0.452489,0.527789,1.000562e-07,13.0


In [4]:
char['DATE'].min(), char['DATE'].max()

(np.int64(19570131), np.int64(20211231))

In [5]:
#create MM-YYYY datetime
char['mdate'] = pd.to_datetime(char['DATE'].astype('str')).dt.to_period('M')
char = char.drop(index=char[char['mdate'] > '2016-12'].index)  #drop mdate > 2016-12 to replicate the paper's results
char = char.sort_values(by=['mdate', 'permno'])  #ensure stock-month order

In [6]:
char_variables = [col for col in char.columns if col not in ['permno', 'DATE', 'mdate', 'sic2']]  #K=94 characteristics
len(char_variables)

94

In [7]:
#check discrete characteristics
char_discrete = []
for c in char_variables:
    if char[c].nunique() <= 2:
        char_discrete.append(c)
        print(f'{c}: {char[c].nunique()} unique values: {char[c].unique()}')

convind: 2 unique values: [nan  0.  1.]
divi: 2 unique values: [nan  0.  1.]
divo: 2 unique values: [nan  0.  1.]
rd: 2 unique values: [nan  0.  1.]
securedind: 2 unique values: [nan  0.  1.]
sin: 2 unique values: [nan  0.  1.]


In [8]:
char_cont = [c for c in char_variables if c not in char_discrete]

In [9]:
#Monthly median values
median = char.groupby('mdate')[char_cont].median().reset_index(drop=False)
for c in char_cont:
    name = f'{c}_med'
    median = median.rename(columns={c: name})
median = median.fillna(0) #for months when all stocks have NaN values, set median=0

In [10]:
#for continuous characteristics: fill nans with monthly median; for discrete: fill nans with 0
for c in char_variables:
    na_idx = char[char[c].isna()].index
    print(f'NA counts for {c} before: {char[c].isna().sum()}')
    if c in char_discrete:
        char[c] = char[c].fillna(0)
    else:
        for d in char.loc[na_idx]['mdate'].unique():
            char.loc[(char[c].isna()) & (char['mdate']==d), c] = median.loc[(median['mdate']==d), f'{c}_med'].item()
    print(f'NA counts for {c} after: {char[c].isna().sum()}')

NA counts for mvel1 before: 3004
NA counts for mvel1 after: 0
NA counts for beta before: 369742
NA counts for beta after: 0
NA counts for betasq before: 369742
NA counts for betasq after: 0
NA counts for chmom before: 311706
NA counts for chmom after: 0
NA counts for dolvol before: 348490
NA counts for dolvol after: 0
NA counts for idiovol before: 369742
NA counts for idiovol after: 0
NA counts for indmom before: 24
NA counts for indmom after: 0
NA counts for mom1m before: 28869
NA counts for mom1m after: 0
NA counts for mom6m before: 143475
NA counts for mom6m after: 0
NA counts for mom12m before: 311706
NA counts for mom12m after: 0
NA counts for mom36m before: 907475
NA counts for mom36m after: 0
NA counts for pricedelay before: 369814
NA counts for pricedelay after: 0
NA counts for turn before: 345296
NA counts for turn after: 0
NA counts for absacc before: 1433581
NA counts for absacc after: 0
NA counts for acc before: 1433581
NA counts for acc after: 0
NA counts for age before: 9

In [10]:
#Download stocks monthly returns from WRDS
#stocklist = np.sort(char['permno'].unique())
#stock_str = ','.join(str(x) for x in stocklist)
#mret = conn.raw_sql(f"""select permco, permno, date, prc, ret, retx, shrout
#                        from crsp.msf
#                        where permno in ({stock_str})
#                        and date>='01/01/1957'
#                        """,
#                     date_cols=['date'])
#mret.to_csv('mret.csv', index=False)
mret = pd.read_csv('mret.csv')
mret['mdate'] = pd.to_datetime(mret['date'].astype(str)).dt.to_period('M')

#align stock-month characteristics and returns
char = pd.merge(char, mret[['mdate','permno','ret']], on=['mdate','permno'])
na_ret = char[char['ret'].isna()].index
char = char.drop(index=na_ret)
char = char.reset_index(drop=True)

In [12]:
#Rank stocks' characteristics each month and map to [-1,1]
def _rank(df, group_by_col, rank_col):
    df = df.copy()
    df['rank'] = df.groupby(group_by_col)[rank_col].rank(method='min', ascending=True)
    max_rank = pd.DataFrame({
        'max_rank': df.groupby(group_by_col)['rank'].max()
    }).reset_index(drop=False)
    df = pd.merge(df, max_rank, on=group_by_col)
    df['rank_map'] = np.where(
        df['max_rank'] - 1 > 0,
        2 * (df['rank'] - 1) / (df['max_rank'] - 1) - 1,
        0.0
    )
    rank_df = df[['mdate', 'permno', 'rank_map']].copy()
    rank_df.rename(columns={'rank_map': rank_col}, inplace=True)
    return rank_df

In [13]:
for c in char_variables:
    if c == char_variables[0]:
        df = char[['mdate', 'permno', c]]
        char_rank = _rank(df, group_by_col='mdate', rank_col=c)
    elif c in char_discrete:
        char_rank = pd.merge(char_rank, char[['mdate', 'permno', c]], on=['mdate', 'permno'])
    else:
        df = char[['mdate', 'permno', c]]
        rank_df = _rank(df, group_by_col='mdate', rank_col=c)
        char_rank = pd.merge(char_rank, rank_df, on=['mdate', 'permno'])

In [14]:
char_rank.head(10)

,mdate,permno,mvel1,beta,betasq,chmom,dolvol,idiovol,indmom,mom1m,...,stdacc,stdcf,ms,baspread,ill,maxret,retvol,std_dolvol,std_turn,zerotrade
0,1957-01,10006,0.275500,0.205005,0.205005,0.366699,0.482194,-0.251203,0.0,0.412897,...,0.0,0.0,0.0,-0.060635,-0.659288,-0.757459,-0.809432,-0.919153,-0.014437,-0.435996
1,1957-01,10014,-0.940896,-0.807507,-0.807507,-0.792108,-0.867180,1.000000,0.0,-0.951877,...,0.0,0.0,0.0,0.940327,0.951877,0.651588,0.944177,0.984601,0.634264,-0.539942
2,1957-01,10022,-0.742612,0.124158,0.124158,-0.039461,-0.688162,-0.085659,0.0,-0.884504,...,0.0,0.0,0.0,0.262753,0.711261,-0.457170,0.274302,0.555342,0.282002,-0.418672
3,1957-01,10030,0.060057,-0.147257,-0.147257,0.233879,0.591915,-0.613090,0.0,0.410972,...,0.0,0.0,0.0,0.162656,-0.487969,0.422522,0.322425,0.301251,0.692012,-0.757459
4,1957-01,10057,-0.090562,0.420597,0.420597,0.272377,-0.376323,-0.280077,0.0,0.711261,...,0.0,0.0,0.0,-0.880654,0.553417,0.782483,0.657363,0.366699,-0.083734,0.707411
5,1957-01,10065,0.227836,0.453321,0.453321,0.799808,0.151107,-0.430221,0.0,0.651588,...,0.0,0.0,0.0,-0.607315,-0.499519,-0.374398,-0.805582,-0.374398,-0.010587,-0.237729
6,1957-01,10102,0.591992,0.697786,0.697786,-0.208855,0.792108,-0.272377,0.0,0.384023,...,0.0,0.0,0.0,-0.518768,-0.749759,-0.640038,-0.790183,0.478345,0.355149,-0.278152
7,1957-01,10137,0.647283,-0.615014,-0.615014,-0.357074,0.143407,-0.788258,0.0,0.091434,...,0.0,0.0,0.0,-0.072185,-0.407122,-0.260828,-0.692012,-0.237729,-0.832531,0.287777
8,1957-01,10145,0.942803,0.235804,0.235804,-0.328200,0.888354,-0.478345,0.0,0.493744,...,0.0,0.0,0.0,0.308951,-0.905679,-0.230029,-0.016362,-0.842156,-0.620789,-0.004812
9,1957-01,10153,0.721640,0.303176,0.303176,0.083734,0.828681,-0.618864,0.0,0.840231,...,0.0,0.0,0.0,0.301251,-0.884504,-0.051011,-0.378248,-0.751684,0.079885,-0.495669


In [15]:
#check if values are in [-1,1]
lower = char_rank.groupby('mdate')[char_variables].min()
upper = char_rank.groupby('mdate')[char_variables].max()
np.where(lower < -1), np.where(upper > 1)

((array([], dtype=int64), array([], dtype=int64)),
 (array([], dtype=int64), array([], dtype=int64)))

In [21]:
#macro predictors data
macros_raw = pd.read_csv(
    f'https://docs.google.com/spreadsheets/d/10_nkOkJPvq4eZgNl-1ys63PzhbnM3S2y/export?format=csv&gid=1922816101')
macros_raw.head(10)

,yyyymm,price,d12,e12,ret,retx,AAA,BAA,lty,ltr,...,ygap,rdsp,rsvix,gpce,gip,tchi,house,avgcor,shtint,disag
0,187101,4.44,0.26,0.4,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,187102,4.50,0.26,0.4,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,187103,4.61,0.26,0.4,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,187104,4.74,0.26,0.4,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,187105,4.86,0.26,0.4,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,187106,4.82,0.26,0.4,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6,187107,4.73,0.26,0.4,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
7,187108,4.79,0.26,0.4,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
8,187109,4.84,0.26,0.4,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
9,187110,4.59,0.26,0.4,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [22]:
#Compute 8 macro predictors
macros_raw['mdate'] = pd.to_datetime(macros_raw['yyyymm'].astype(str), format='%Y%m').dt.to_period('M')
macro = macros_raw.copy()

#dividend price ratio = log(dividends/lagged prices)
macro['dp'] = np.log(macro['d12'] / macro['price'].shift(1))

#earnings price ratio = log(earnings/ lagged prices)
macro['e/p'] = np.log(macro['e12'] / macro['price'].shift(1))
#term spread = long term yield on gov bond - treasury bill
macro['tms'] = macro['lty'] - macro['tbl']

#default yield spread = BAA - AAA corp bond yields
macro['dfy'] = macro['BAA'] - macro['AAA']

macro = macro[['mdate', 'dp', 'e/p', 'b/m', 'ntis', 'tbl', 'tms', 'dfy', 'svar']]
macro_variables = ['dp', 'e/p', 'b/m', 'ntis', 'tbl', 'tms', 'dfy', 'svar']

#shift to create lagged predictors
macro[macro_variables] = macro[macro_variables].shift(1)
macro = macro.drop(index=macro[(macro['mdate'] < '1957-01') | (macro['mdate'] > '2016-12')].index)
macro = macro.reset_index(drop=True)
macro.head()

,mdate,dp,e/p,b/m,ntis,tbl,tms,dfy,svar
0,1957-01,-3.254554,-2.581726,0.544177,0.026150,0.0321,0.0024,0.0062,0.001020
1,1957-02,-3.291115,-2.617357,0.567243,0.027993,0.0311,0.0017,0.0072,0.000902
2,1957-03,-3.250394,-2.575675,0.584994,0.030174,0.0310,0.0018,0.0080,0.001056
3,1957-04,-3.219107,-2.543453,0.599819,0.026601,0.0308,0.0023,0.0077,0.000330
4,1957-05,-3.238565,-2.560942,0.576098,0.027422,0.0307,0.0038,0.0077,0.000302


In [ ]:
#Compute excess returns
char = pd.merge(char, macros_raw[['mdate','Rfree']], on=['mdate'])
char['excessret'] = char['ret'] - char['Rfree']

#merge excess returns, ranked characteristics, sic dummies & macros to 1 df
X = pd.merge(char_rank, char[['mdate','permno','sic2','excessret']], on=['mdate','permno'])
X = pd.merge(X, macro, on='mdate')

In [25]:
#check the numbers of predictors
len(macro_variables), len(char_variables)

(8, 94)

In [26]:
sic_categories = [np.sort(X['sic2'].dropna().unique())]
sic_ohe = OneHotEncoder(handle_unknown='ignore', categories=sic_categories, sparse_output=False)
sic_categories[0].shape[0]

74

In [27]:
n_interactions = (len(macro_variables) + 1) * len(char_variables)
n_signals = n_interactions + sic_categories[0].shape[0]
n_interactions, n_signals

(846, 920)

In [28]:
#Compute interaction terms
def _interaction(df):
    chars = df[char_variables].to_numpy(dtype='float32')
    macros = df[macro_variables].to_numpy(dtype='float32')
    z = [chars * macros[:, [i]] for i in range(macros.shape[1])]
    z_stacked = np.hstack(z)
    sic = sic_ohe.fit_transform(df[['sic2']])
    z_stacked = np.hstack((z_stacked, sic))
    gc.collect()
    return np.hstack((df[char_variables],z_stacked))

In [29]:
#Save design matrix and target variables as memmap files
Xdata = np.memmap(path.join(os.getcwd(), 'Xdata.dat'), dtype='float32', mode='w+', shape=(X.shape[0], n_signals))
ydata = np.memmap(path.join(os.getcwd(), 'ydata.dat'), dtype='float32', mode='w+', shape=(X.shape[0], 1))

for i in range(0, X.shape[0], 10000):
    end = min(i + 10000, X.shape[0])
    X_df = X.iloc[i:end]
    z = _interaction(X_df)
    Xdata[i:end] = z[:]

ydata[:] = X[['excessret']][:]

((3743049, 920), (3743049, 1))

In [33]:
#Create stopping indices for training, validation & testing samples
trainperiod = pd.to_datetime(pd.date_range(start='1974-12-31', end='2003-12-31', freq='YE')).to_period('M')
idx = []
for j in trainperiod:
    train = X[X['mdate'] <= j].shape[
        0]  #training samples; first training sample = 18 years; recursively increase by 1 year
    val = train + X[X['mdate'].between(j + 1, j + 12*12)].shape[
        0]  #validation samples = 12 years after training sample
    test = val + X[X['mdate'].between(j + 12*12 + 1, j + 13*12)].shape[0]  #testing samples = 1 year after validation sample
    idx.append((train, val, test))
idx = pd.DataFrame(idx, columns=['train', 'val', 'test'])
idx #for example: the first split has training sample indices go from 0 to 478008, validation sample from 478009 to 1248578, testing sample from 1248579 to 1331524

,train,val,test
0,478009,1248579,1331524
1,536573,1331524,1415709
2,595435,1415709,1497574
3,654676,1497574,1578357
4,712648,1578357,1658146
5,770087,1658146,1739760
6,827998,1739760,1826302
7,889836,1826302,1921933
8,953815,1921933,2020327
9,1022196,2020327,2124262


In [35]:
idx = idx.to_numpy()
char_test = char.iloc[idx[0][1]:].reset_index(drop=True)

#get testing sample monthly periods
oos_periods = np.array(char_test['mdate'])

#get testing sample stock id
permno = np.array(char_test['permno'])

#get testing sample market values
mktval = np.array(char_test['mvel1'])

#get indices of the top 1000 and bottom 1000 stocks by market value from testing sample
mvel_rank = char_test.groupby('mdate')['mvel1'].rank(method='min',ascending=True)
mvel_max_rank = mvel_rank.groupby(char_test['mdate']).transform('max')
top1000 = mvel_rank[mvel_rank > mvel_max_rank - 1000].index
bot1000 = mvel_rank[mvel_rank < 1000].index

In [36]:
#Save files
idx.to_csv('idx.csv', index=False)
np.save('top1000.npy', top1000)
np.save('bot1000.npy', bot1000)
np.save('permno.npy', permno)
np.save('mvel.npy', mktval)
np.save('oos_periods.npy', oos_periods)
np.save('char_variables.npy', char_variables)
np.save('macro_variables.npy', macro_variables)